# Visual document RAG experiments

Set up Colab, fill the CSV, choose settings in config.json, then run both experiments with one command. Run folders and resuming are automatic.


In [ ]:
!nvidia-smi


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
%cd /content/drive/MyDrive/visual_rag


In [ ]:
# Keep the package versions from your working notebook.
# Run the next line ONLY if this is a fresh environment missing packages.
# %pip install -r requirements.txt


## Document folders and PostgreSQL

Put originals in data/docs/clean/. Put each modified PDF in cell/, header/, random_cell/, or random_header/ as appropriate. Unedited copies are generated in duplicates/. The following setup follows the PostgreSQL/pgvector setup from your working notebook.

In [ ]:
%%bash
set -e
apt-get update -qq
apt-get install -y postgresql postgresql-server-dev-all build-essential git
service postgresql start
if [ ! -d /content/pgvector ]; then
    git clone --branch v0.8.1 https://github.com/pgvector/pgvector.git /content/pgvector
fi
cd /content/pgvector
make
make install
sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'colab_local';"


In [ ]:
import os
os.environ["PGPASSWORD"] = "colab_local"  # local Colab database account
os.environ["PGUSER"] = "postgres"


In [ ]:
from pathlib import Path
for name in ("clean", "duplicates", "cell", "header", "random_cell", "random_header", "chart", "layout", "random_chart", "random_layout"):
    (Path("data/docs") / name).mkdir(parents=True, exist_ok=True)
Path("results").mkdir(exist_ok=True)
# Copy existing originals from data/docs/non_pois/ into data/docs/clean/ if needed.
# Classify existing poisoned PDFs by their actual edit type before placing them.


## 1. Prepare the data and settings

Fill data/questions.csv. Put PDFs in the matching document folders. The first five columns are sufficient for a clean pilot; Fill the columns for each selected attack and its matched random control. Chart/layout fields are optional when those attacks are not selected. Set attacks and model/retrieval settings in config.json. The default is cell-only.

Include all selected documents plus distractors in clean/ (50 clean pages recommended for top-10 retrieval). Cell and header comparisons must use the same edited page and wrong target. Experiment A passes 5 pages to the generator by default; B keeps its condition-specific context.

In [ ]:
import json
from pathlib import Path
import shutil
Path("data").mkdir(exist_ok=True)
if not Path("data/questions.csv").exists():
    shutil.copyfile("questions_template.csv", "data/questions.csv")
cfg = json.loads(Path("config.json").read_text())
print("Attacks:", cfg.get("attacks", ["cell"]))
print("Retrieval top-k:", cfg["retrieval_k"], "Final context pages:", cfg["context_pages"])
print("Input:", cfg["csv"])


## Optional: create chart/layout PDFs

Use attacks.py to inspect regions and create PDF variants. Read ATTACKS.md and fill data/attack_edits.json from the example before building. The commands below are commented until you provide actual filenames and edit annotations. Select chart/layout in config.json and fill their CSV fields before the full experiment.

In [ ]:
# !python attacks.py inspect --doc YOUR_REPORT.pdf --page 3
# !python attacks.py build --edits data/attack_edits.json


## 2. Optional clean pilot

Start with 1–2 questions. No edited PDFs are needed. This runs clean/duplication A conditions and no-context/clean B conditions.

In [ ]:
!python run_experiment_a.py --both --clean --validate-only


In [ ]:
!python run_experiment_a.py --both --clean


## 3. Run both experiments

This uses the attacks selected in config.json. Repeating the command resumes matching results. Changing settings or data starts a separate run automatically. No run names are required.

Completed question/condition outputs are reused across runs. Adding only questions reuses old work; changing the retrieval corpus requires rerunning affected A cases. Editing evaluate.py or entering manual review labels does not rerun inference.


In [ ]:
!python run_experiment_a.py --both


## 4. Review the latest answers manually

The next cell finds the latest run for A and B and creates review copies without overwriting existing labels. Open each reviewed_answers.csv and review the full prediction alongside the question, reference answers, and context_pages.

Enter 1/0 in correct for completed answers. For targeted attacks only, enter 1/0 in target_match to indicate whether the answer gives the predefined wrong answer. Leave target_match blank for baseline and random-edit rows. Leave both labels blank for failed/incomplete executions.

Blank labels stay unreviewed. There is no automatic answer matching or scoring. The Final answer: format is only part of the unchanged model prompt; you decide the label.


In [ ]:
import json
import shutil
from pathlib import Path
cfg = json.loads(Path("config.json").read_text())
results = Path(cfg["results_dir"])
for experiment in ("a", "b"):
    pointer = results / f"latest_{experiment}.json"
    if not pointer.exists():
        print(f"No experiment {experiment.upper()} results yet.")
        continue
    saved = json.loads(pointer.read_text())
    folder = results / saved["run_dir"]
    source = folder / "review_template.csv"
    target = folder / "reviewed_answers.csv"
    if source.exists() and not target.exists():
        shutil.copyfile(source, target)
    print(f"{experiment.upper()}: {saved['completed']}/{saved['total']} completed")
    print("Review:", target)


## 5. Summarize your labels and page exposure

These commands use the latest A/B run and load reviewed_answers.csv if present. Answer accuracy stays blank until all completed answers in a condition have correctness labels. Paired accuracy drop and ASR wait for the necessary paired executions and labels. Partial reviewed-only figures are named separately. Check incomplete and unreviewed counts before reporting.

For Experiment A, compare tracked_cell_context_rate (or tracked_header_context_rate) in A1, the corresponding attack, and its matched A4 control. These track the same added page, rather than any page in the inserted document. The corresponding retrieval columns end in retrieval_rate_at_1, _5, and _k. A clean-only pilot reports added-document exposure, since it has no validated attack target pages.


In [ ]:
!python evaluate.py --experiment a
!python evaluate.py --experiment b


In [ ]:
import json
from pathlib import Path
from IPython.display import display
import pandas as pd

results = Path(json.loads(Path("config.json").read_text())["results_dir"])
for experiment in ("a", "b"):
    pointer = results / f"latest_{experiment}.json"
    if not pointer.exists():
        continue
    folder = results / json.loads(pointer.read_text())["run_dir"]
    summary_path = folder / "summary.csv"
    print(f"\nExperiment {experiment.upper()}")
    if summary_path.exists():
        print("Main results:", summary_path)
        display(pd.read_csv(summary_path, keep_default_na=False))
    else:
        print("Generate summary.csv by running the evaluate.py cell above.")


## Optional code checks

These use fake model/database responses where necessary; their outputs are not research results.

In [ ]:
!python -m unittest discover -s tests -v
